# Image Retrieval Model - Complete Training Pipeline (Colab)

This notebook provides a complete training pipeline for an image retrieval model using:
- **GPU Acceleration**: NVIDIA GPU via Google Colab
- **FAISS Vector Store**: Fast similarity search on embeddings
- **Pre-trained Models**: ResNet50 or ViT for image encoding
- **Auto Download**: Model and results automatically downloaded at the end

**Time**: ~1-3 hours depending on dataset size  
**GPU**: Required (enable in Runtime → Change runtime type → GPU)  
**Storage**: ~5-10GB for COCO dataset (adjustable)

## Quick Start:
1. Run cells from top to bottom
2. Enable GPU when prompted
3. Download model at the end

In [ ]:
# Step 1: Setup - Mount Google Drive and check GPU
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

# Create project directory
project_dir = '/content/drive/MyDrive/Image-Retrieval-Model'
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

# Check GPU
import torch
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda')
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → Select 'GPU'")
    device = torch.device('cpu')

In [ ]:
# Step 2: Install dependencies
print("Installing dependencies (this may take 2-3 minutes)...")
!pip install -q torch torchvision transformers pillow tqdm numpy requests faiss-gpu opencv-python scikit-learn

import faiss
print(f"\n✓ FAISS installed successfully")
print(f"✓ FAISS GPU devices: {faiss.get_num_gpus()}")

print("\n✓ All dependencies installed!")

In [ ]:
# Step 3: Create FAISS Vector Store with GPU support
import os
import json
import numpy as np
import faiss
from pathlib import Path
from typing import List, Dict, Optional

class FAISSVectorStore:
    """FAISS-based vector store with GPU support"""
    def __init__(self, embedding_dim: int = 512, use_gpu: bool = True):
        self.embedding_dim = embedding_dim
        self.use_gpu = use_gpu and faiss.get_num_gpus() > 0
        self.metadata = []
        
        if self.use_gpu:
            res = faiss.StandardGpuResources()
            self.index = faiss.GpuIndexFlatL2(res, embedding_dim)
            print("✓ Using GPU FAISS")
        else:
            self.index = faiss.IndexFlatL2(embedding_dim)
            print("✓ Using CPU FAISS")
    
    def add_embeddings(self, embeddings: np.ndarray, image_names: List[str]) -> None:
        if embeddings.shape[1] != self.embedding_dim:
            raise ValueError(f"Dimension mismatch: {embeddings.shape[1]} vs {self.embedding_dim}")
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings.astype(np.float32))
        self.metadata.extend(image_names)
        print(f"✓ Added {len(embeddings)} embeddings (Total: {self.index.ntotal})")
    
    def search(self, query_embedding: np.ndarray, k: int = 10) -> List[Dict]:
        if query_embedding.ndim == 1:
            query_embedding = query_embedding.reshape(1, -1)
        faiss.normalize_L2(query_embedding)
        distances, indices = self.index.search(query_embedding.astype(np.float32), k)
        
        results = []
        for rank, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            if idx >= 0:
                results.append({
                    'rank': rank + 1,
                    'image_name': self.metadata[idx],
                    'distance': float(distance),
                    'similarity': 1.0 / (1.0 + float(distance))
                })
        return results
    
    def save_index(self, index_path: str, metadata_path: str) -> None:
        os.makedirs(os.path.dirname(index_path) or '.', exist_ok=True)
        if self.use_gpu:
            cpu_index = faiss.index_gpu_to_cpu(self.index)
            faiss.write_index(cpu_index, index_path)
        else:
            faiss.write_index(self.index, index_path)
        
        metadata = {
            'embedding_dim': self.embedding_dim,
            'num_vectors': self.index.ntotal,
            'image_names': self.metadata
        }
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f)
        print(f"✓ Saved index to {index_path}")
    
    def load_index(self, index_path: str, metadata_path: str) -> None:
        cpu_index = faiss.read_index(index_path)
        if self.use_gpu:
            res = faiss.StandardGpuResources()
            self.index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        else:
            self.index = cpu_index
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)['image_names']
        print(f"✓ Loaded index from {index_path}")
    
    def get_size(self) -> int:
        return self.index.ntotal

# Initialize vector store
os.makedirs('vector_store', exist_ok=True)
vector_store = FAISSVectorStore(embedding_dim=2048, use_gpu=True)

In [ ]:
# Step 4: Download COCO Dataset (sample for faster training)
print("Preparing dataset...")

# Create directories
os.makedirs('dataset/coco/train2017', exist_ok=True)
os.makedirs('dataset/coco/annotations', exist_ok=True)

# Download annotations
print("\nDownloading COCO annotations...")
!cd dataset/coco/annotations && wget -q https://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip 2>/dev/null || true

# Download sample images (using COCO API to get valid images)
import json
from PIL import Image
import io
import urllib.request

print("\nDownloading sample COCO images...")

# Load annotations to get image info
try:
    with open('dataset/coco/annotations/instances_train2017.json', 'r') as f:
        data = json.load(f)
    images = data['images'][:200]  # Use first 200 images
    print(f"Found {len(images)} images in annotations")
except:
    images = []
    print("Creating sample dataset...")

# Download images
downloaded_count = 0
for i, img_info in enumerate(images[:200]):
    try:
        img_id = img_info['id']
        img_url = f"http://images.cocodataset.org/train2017/{img_id:012d}.jpg"
        img_path = f"dataset/coco/train2017/{img_id:012d}.jpg"
        
        if not os.path.exists(img_path):
            urllib.request.urlretrieve(img_url, img_path)
            downloaded_count += 1
        
        if (i + 1) % 20 == 0:
            print(f"Downloaded {downloaded_count}/{i+1} images...", end='\r')
    except Exception as e:
        pass

# Create sample images if download fails
if len(os.listdir('dataset/coco/train2017')) < 50:
    print("\nCreating synthetic sample images...")
    import random
    for i in range(100):
        img = Image.new('RGB', (224, 224), color=(random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)))
        img.save(f'dataset/coco/train2017/sample_{i:06d}.jpg')

image_files = os.listdir('dataset/coco/train2017')
print(f"\n✓ Dataset ready: {len(image_files)} images")

In [ ]:
# Step 6: Generate embeddings for all images
print("Generating embeddings for all images...")
print(f"Processing {len(dataset)} images in batches of 16...\n")

all_embeddings = []
all_image_names = []

with torch.no_grad():
    for batch_idx, (images, image_names) in enumerate(tqdm(dataloader)):
        images = images.to(device)
        
        # Generate embeddings
        embeddings = model(images)
        embeddings = embeddings.cpu().numpy()
        
        all_embeddings.append(embeddings)
        all_image_names.extend(image_names)
        
        if (batch_idx + 1) % 5 == 0:
            print(f"  Processed {min((batch_idx + 1) * 16, len(dataset))}/{len(dataset)} images")

# Combine all embeddings
all_embeddings = np.vstack(all_embeddings)
print(f"\n✓ Generated {len(all_embeddings)} embeddings")
print(f"  Shape: {all_embeddings.shape}")
print(f"  Memory: {all_embeddings.nbytes / 1e9:.2f} GB")

In [ ]:
# Step 5: Load Pre-trained Model (ResNet50)
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torch.nn as nn

print("Loading ResNet50 model...")
model = models.resnet50(pretrained=True)
model.fc = nn.Identity()  # Remove classification layer
model = model.to(device)
model.eval()

print(f"✓ Model loaded (output dim: 2048)")

# Define image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Create dataset class
class ImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        self.transform = transform
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.image_files[idx]
        except:
            # Return black image on error
            return torch.zeros(3, 224, 224), self.image_files[idx]

# Create dataset and dataloader
dataset = ImageDataset('dataset/coco/train2017', transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"✓ Dataset loaded: {len(dataset)} images")

In [ ]:
# Step 7: Add embeddings to FAISS index
print("Adding embeddings to FAISS index...\n")
vector_store.add_embeddings(all_embeddings, all_image_names)
print(f"\n✓ Vector store now contains {vector_store.get_size()} embeddings")

In [ ]:
# Step 8: Test similarity search
print("Testing similarity search...\n")

# Pick a random query image
query_idx = np.random.randint(0, len(all_embeddings))
query_embedding = all_embeddings[query_idx:query_idx+1]

print(f"Query image: {all_image_names[query_idx]}\n")
print("Top 10 similar images:")
print("-" * 60)

results = vector_store.search(query_embedding, k=10)
for result in results:
    print(f"  Rank {result['rank']}: {result['image_name']:<40} | "
          f"Similarity: {result['similarity']:.4f} | Distance: {result['distance']:.4f}")

In [ ]:
# Step 9: Save model and vector store
import shutil
from datetime import datetime

print("Saving model and vector store...\n")

# Create results directory
results_dir = 'trained_model'
os.makedirs(results_dir, exist_ok=True)

# Save the ResNet model
model_path = os.path.join(results_dir, 'resnet50_encoder.pth')
torch.save(model.state_dict(), model_path)
print(f"✓ Saved model to {model_path}")

# Save FAISS index
index_path = os.path.join(results_dir, 'image_embeddings.index')
metadata_path = os.path.join(results_dir, 'image_embeddings_metadata.json')
vector_store.save_index(index_path, metadata_path)

# Save configuration
config = {
    'model_type': 'ResNet50',
    'embedding_dim': 2048,
    'num_images': len(all_image_names),
    'num_vectors': vector_store.get_size(),
    'timestamp': datetime.now().isoformat()
}
config_path = os.path.join(results_dir, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Saved config to {config_path}")

print(f"\n✓ Training complete!")
print(f"  Model saved to: {results_dir}/")
print(f"  Total embeddings: {vector_store.get_size()}")
print(f"  Embedding dimension: 2048")

In [ ]:
# Step 10: Download model (automatic download)
print("Preparing model for download...\n")

# Create zip file
import zipfile
zip_filename = 'Image-Retrieval-Model.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for root, dirs, files in os.walk('trained_model'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '.')
            zipf.write(file_path, arcname)
            print(f"  Added: {arcname}")

print(f"\n✓ Model package ready: {zip_filename}")

# Trigger download using Colab's download function
from google.colab import files
print("\nDownloading to your computer...")
files.download(zip_filename)
print("\n✓ Download complete! Check your Downloads folder.")

# Also save to Google Drive
drive_zip = f'/content/drive/MyDrive/{zip_filename}'
shutil.copy(zip_filename, drive_zip)
print(f"✓ Also saved to Google Drive: {drive_zip}")

## Training Complete!

Your model has been trained and is ready to download. The package includes:
- **resnet50_encoder.pth** - Pre-trained ResNet50 encoder weights
- **image_embeddings.index** - FAISS vector index (for fast similarity search)
- **image_embeddings_metadata.json** - Image names and metadata
- **config.json** - Training configuration

### How to Use the Downloaded Model

```python
import torch
import faiss
import json

# Load the model
model = torch.load('resnet50_encoder.pth')

# Load FAISS index
index = faiss.read_index('image_embeddings.index')
with open('image_embeddings_metadata.json', 'r') as f:
    metadata = json.load(f)

# Search for similar images
query_embedding = model(your_image)  # Shape: (1, 2048)
distances, indices = index.search(query_embedding, k=5)
print([metadata['image_names'][i] for i in indices[0]])
```

---

## Alternative Training Methods

### 1. **Transfer Learning with Fine-tuning** ✓ Fastest
- Use pre-trained CLIP, ViT, or ResNet (current method)
- Pros: Fast, good results, requires less data
- Cons: Limited to pre-trained architecture
- Time: 30 mins - 2 hours

### 2. **Contrastive Learning (SimCLR)**
```bash
python train_simclr.py --dataset coco --epochs 50 --batch-size 128
```
- Pros: Better generalization, self-supervised
- Cons: Requires more computation time
- Time: 6-12 hours
- GPU Memory: 16GB+

### 3. **Metric Learning (Triplet Loss)**
```bash
python train_triplet.py --dataset coco --epochs 30 --margin 1.0
```
- Pros: Optimized for similarity search
- Cons: Needs hard negative mining
- Time: 3-6 hours
- GPU Memory: 12GB+

### 4. **CLIP Fine-tuning**
```bash
python train_clip.py --dataset coco --epochs 10 --lr 1e-5
```
- Pros: Multimodal (image + text), very good results
- Cons: Slower training
- Time: 4-8 hours
- GPU Memory: 20GB+

### 5. **Distributed Training with PyTorch Lightning**
```bash
python -m pytorch_lightning.cli fit --model resnet50 --trainer.gpus 4
```
- Pros: Multi-GPU support, faster training
- Cons: More complex setup
- Time: 1-2 hours (on 4 GPUs)
- GPU Memory: 8GB per GPU

### 6. **Lightweight Models for Mobile**
- Use MobileNetV3, EfficientNet, or SqueezeNet
- Pros: Fast inference, small model size
- Cons: Slightly lower accuracy
- Time: 15-30 mins

### 7. **Local Training on Your Machine**
```bash
# Activate venv
.venv\Scripts\Activate.ps1

# Run training
python train_model.py --batch-size 16 --epochs 10
```
- Pros: No GPU fees, full control
- Cons: Very slow (CPU only)
- Time: 24-48 hours
- GPU Memory: Not needed

### 8. **AWS SageMaker Training**
- Pros: Distributed training, managed infrastructure
- Cons: Paid service
- Cost: $1-5/hour

---

## Recommended Setup Comparison

| Method | Time | GPU Needed | Cost | Quality |
|--------|------|-----------|------|---------|
| **Colab + Transfer Learning** (Current) | 1-2h | Free | Free | ⭐⭐⭐⭐ |
| **Local CPU** | 24-48h | No | Free | ⭐⭐⭐ |
| **SimCLR on Colab** | 6-12h | Free | Free | ⭐⭐⭐⭐⭐ |
| **CLIP Fine-tune** | 4-8h | Free | Free | ⭐⭐⭐⭐⭐ |
| **AWS SageMaker** | 2-4h | Yes | $3-10 | ⭐⭐⭐⭐⭐ |
| **Triplet Loss** | 3-6h | Free | Free | ⭐⭐⭐⭐ |

---

## Quick Setup for Other Methods (Locally)

**Install full dependencies:**
```bash
pip install -r requirements.txt torch torchvision pytorch-lightning transformers faiss-cpu pillow tqdm
```

**Run fine-tuning:**
```bash
python train_with_faiss.py --dataset dataset/coco --epochs 5 --batch-size 8
```

**View training logs:**
```bash
tensorboard --logdir=./logs
```

## Quick Reference: Running This Notebook

### Step-by-Step:
1. **Open this notebook in Colab:**
   - Upload to colab.research.google.com
   - OR open directly if shared as link

2. **Enable GPU:**
   - Runtime → Change runtime type → Select "GPU"

3. **Run all cells:**
   - Cells menu → Run all cells
   - Or press Ctrl+F9

4. **Monitor progress:**
   - Check output after each cell
   - GPU usage in Step 11 (optional)

5. **Download model:**
   - Automatic download popup in Step 10
   - Also saved to Google Drive

### Troubleshooting

**"No GPU" error:**
- Go to Runtime → Change runtime type → Select GPU → Save

**Out of memory:**
- Reduce batch_size (in Step 5, change 16 to 8 or 4)
- Reduce number of images (change 200 to 100)

**Dataset download slow:**
- Already handled - uses existing annotations
- Creates synthetic images if download fails

**Model not saving:**
- Check Google Drive quota (need ~2GB free)
- Save to local Colab `/tmp/` instead

### Files Generated
- `trained_model/` - All model files
- `Image-Retrieval-Model.zip` - Downloadable package
- `vector_store/` - FAISS indices
- `dataset/coco/` - Downloaded images

### Total Time Estimate
- GPU Setup: 2 mins
- Dependencies: 3 mins
- Dataset Download: 5-10 mins
- Embedding Generation: 10-15 mins
- **Total: 20-30 minutes**